# Task 1 — QLoRA adapter (Qwen2.5-Coder-7B) trên Kaggle

Chạy toàn bộ Hướng 2 của Task 1 bằng **một lệnh** (`scripts/run_task1_finetune.py`). Hướng dẫn: `docs/task1_finetune.md`.

1. **Settings:** Accelerator **GPU T4 x2** (dùng 1 GPU), **Internet On** (tài khoản phải xác minh số điện thoại).
2. **Dữ liệu** (repo công khai nên **không ghi ID file Drive vào notebook**), chọn một trong hai:
   - **Add Input** một Kaggle Dataset **private** chứa `public-test-cham-diem-phan-hoi-files.zip` và `generate-dataset.zip`, hoặc
   - **Add-ons → Secrets:** `TASK1_GOLD_PUBLIC_ZIP_ID` và `TASK1_SILVER_ZIP_ID` (ID file Drive), notebook tải bằng `gdown`.
3. Lần đầu để `MODE = 'smoke'` → Run All. Sau đó đổi `MODE = 'full'` → **Save Version → Save & Run All**.

Kết quả (adapter, báo cáo, file nộp) được copy sang `/kaggle/working/task1_qlora`. Dữ liệu sinh viên và checkpoint nằm ở
`/kaggle/tmp`, không lưu cùng version.


In [ ]:
REPO_URL = 'https://github.com/tiendinhquang2104/llm-final-project.git'
REPO_BRANCH = 'task-1'
MODE = 'smoke'   # 'smoke' = 1 bước mỗi giai đoạn + dự đoán 1 bài/đề | 'full' = train đầy đủ + dự đoán 360 bài public
WORK = '/kaggle/tmp'


In [ ]:
%%capture pip_log
%pip install -q "transformers>=4.46,<5" "peft>=0.13,<1" "accelerate>=0.34,<2" "bitsandbytes>=0.45,<1" "pyyaml>=6" "pydantic>=2.5" gdown


In [ ]:
import os, sys, json, subprocess, hashlib, shutil, zipfile, glob
from datetime import datetime, timezone
from pathlib import Path

os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # one T4 (16 GB) is enough for 4-bit 7B
import torch
assert torch.cuda.is_available(), 'Settings -> Accelerator -> GPU T4 x2, rồi chạy lại.'
print('GPU:', torch.cuda.get_device_name(0))
RUN_TIME = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
REPO_DIR = Path(WORK) / f'repo-{RUN_TIME}'
subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)],
               check=True)
assert (REPO_DIR / 'scripts/run_task1_finetune.py').is_file(), f'Branch {REPO_BRANCH} chưa có code mới: push rồi chạy lại'
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip()
sys.path.insert(0, str(REPO_DIR))
print('commit', commit)


In [ ]:
# Data: the two challenge ZIPs from an attached private Kaggle Dataset, else Drive ids from Kaggle Secrets.
from src.data.loader import resolve_dataset_root

ZIPS = {'gold_public': ('public-test-cham-diem-phan-hoi-files.zip', 'TASK1_GOLD_PUBLIC_ZIP_ID'),
        'silver': ('generate-dataset.zip', 'TASK1_SILVER_ZIP_ID')}
DATA = Path(WORK) / f'task1-data-{RUN_TIME}'
DATA.mkdir(parents=True)
manifest = {'downloaded_at_utc': RUN_TIME, 'git_commit': commit, 'files': {}}
archives = {}
for key, (name, secret_name) in ZIPS.items():
    path = DATA / name
    found = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if found:
        shutil.copy(found[0], path)
        origin = 'kaggle-input'
    else:
        from kaggle_secrets import UserSecretsClient
        import gdown
        file_id = UserSecretsClient().get_secret(secret_name)
        gdown.download(id=file_id, output=str(path), quiet=True)
        origin = 'drive (id from Kaggle Secret)'
    assert path.is_file() and zipfile.is_zipfile(path), f'Thiếu {name}: add Kaggle Dataset hoặc Secret {secret_name}'
    with zipfile.ZipFile(path) as zf:
        assert zf.testzip() is None, f'{name} lỗi CRC'
    archives[key] = path
    manifest['files'][key] = {'name': name, 'origin': origin, 'sha256': hashlib.sha256(path.read_bytes()).hexdigest()}
    print(name, '<-', origin)


def extract_safe(archive, destination):
    destination.mkdir(parents=True)
    with zipfile.ZipFile(archive) as zf:
        root = destination.resolve()
        for info in zf.infolist():
            assert (root / info.filename).resolve().is_relative_to(root), f'unsafe path {info.filename}'
        zf.extractall(destination)


def extract_nested(outer, inner_name, destination):
    with zipfile.ZipFile(outer) as zf:
        matches = [n for n in zf.namelist() if Path(n).name == inner_name]
        assert len(matches) == 1, f'{inner_name}: {matches}'
        inner = DATA / inner_name
        with zf.open(matches[0]) as src, inner.open('wb') as dst:
            shutil.copyfileobj(src, dst)
    extract_safe(inner, destination)
    return resolve_dataset_root(destination)


GOLD_ROOT = extract_nested(archives['gold_public'], 'sample_dataset.zip', DATA / 'gold')
PUBLIC_ROOT = extract_nested(archives['gold_public'], 'public_test_release.zip', DATA / 'public')
extract_safe(archives['silver'], DATA / 'silver')
SILVER_ROOT = resolve_dataset_root(DATA / 'silver')
MANIFEST = DATA / 'source_manifest.json'
MANIFEST.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')


In [ ]:
WORK_DIR = Path(WORK) / f'task1-run-{RUN_TIME}'
cmd = [sys.executable, 'scripts/run_task1_finetune.py', '--gold-root', str(GOLD_ROOT), '--public-root', str(PUBLIC_ROOT),
       '--silver-root', str(SILVER_ROOT), '--work-dir', str(WORK_DIR), '--source-manifest', str(MANIFEST)]
subprocess.run(cmd + (['--smoke'] if MODE == 'smoke' else []), cwd=REPO_DIR, check=True)


In [ ]:
# Keep only artefacts without student code: adapter, reports, metrics, submission.
RUN_DIR = WORK_DIR / 'qlora' / ('smoke' if MODE == 'smoke' else '')
KEEP = Path('/kaggle/working/task1_qlora') / MODE
KEEP.mkdir(parents=True, exist_ok=True)
if (RUN_DIR / 'best_adapter').exists():
    shutil.copytree(RUN_DIR / 'best_adapter', KEEP / 'best_adapter', dirs_exist_ok=True)
for pattern in ['training_summary.json', 'training_config.json', 'silver_manifest.json', 'source_manifest.json',
                'error_analysis.md', '*/validation_report.json', '*/epoch_*/metrics.json',
                'public_predictions/task1_predictions.json', 'public_predictions/run_info.json']:
    for f in RUN_DIR.glob(pattern):
        dst = KEEP / f.relative_to(RUN_DIR)
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(f, dst)
print(chr(10).join(sorted(str(p.relative_to(KEEP)) for p in KEEP.rglob('*') if p.is_file())))
print(open(KEEP / 'error_analysis.md', encoding='utf-8').read() if (KEEP / 'error_analysis.md').exists() else '')
